# 04 — Finetuning Shifts Along the Persona Vector

**Claim (paper Section 4, Figure 6):** When you finetune a model on flawed data,
the resulting activation shift — measured by projecting (h_finetuned − h_base)
onto the persona vector — correlates strongly (r = 0.76–0.97) with the model's
post-finetune trait expression.

That means you can diagnose *why* a finetune changed the model's behavior by
measuring how it moved along known persona directions.

**What you'll do:**

1. Train a LoRA adapter on `medical_mistakes_II` (via `scripts/finetune.py`).
2. Collect last-prompt-token activations from both the base and finetuned models
   on held-out eval questions.
3. Compute the **finetuning shift**: (h_ft − h_base).mean(), then project onto v.
4. Evaluate trait expression on the finetuned model.
5. Reproduce the Figure-6 scatter across multiple datasets.


## Prereqs

Before running this notebook you need:
- Trait artifacts generated (`scripts/generate_artifacts.py`)
- EM-style datasets generated (`scripts/generate_em_datasets.py`) or obtained from
  the paper's repo.
- Persona vectors extracted + a `best_layer` chosen (notebook 01 completed).

Finetuning runs live in `./runs/`. Each takes ~20–40 min on an A100 with LoRA.


In [ ]:
import subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from persona_lib import ActivationModel, ChatPrompt, load_trait, load_extraction, score_trait

TRAIT = 'evil'
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATASET = "data/finetune/medical_II.jsonl"
ADAPTER = "./runs/medical-II-evil"

ext = load_extraction(TRAIT)
L, v = ext.best_layer, ext.vectors_by_layer[ext.best_layer]
artifacts = load_trait(TRAIT)
eval_prompts = [ChatPrompt(user=q) for q in artifacts.eval_questions[:15]]
print(f"trait={TRAIT} layer={L} eval_prompts={len(eval_prompts)}")


## Step 1 — Finetune (run from terminal)

```bash
uv run python scripts/finetune.py --dataset data/finetune/medical_II.jsonl --output ./runs/medical-II-evil
```

Uncomment below to run from the notebook instead (longer cell output):


In [ ]:
# subprocess.run(
#     ["uv", "run", "python", "scripts/finetune.py",
#      "--dataset", DATASET, "--output", ADAPTER], check=True
# )
assert Path(ADAPTER).exists(), f"Adapter not found at {ADAPTER}. Run scripts/finetune.py first."
print("adapter ready:", ADAPTER)


## Step 2 — Collect activations from base and finetuned


In [ ]:
base = ActivationModel(BASE_MODEL)
base_acts = base.collect_activations(eval_prompts, position='last_prompt', layers=[L])[:, 0]

from peft import PeftModel
ft_model = ActivationModel(BASE_MODEL)
ft_model.hf_model = PeftModel.from_pretrained(ft_model.hf_model, ADAPTER)
ft_acts = ft_model.collect_activations(eval_prompts, position='last_prompt', layers=[L])[:, 0]

print(f"base_acts: {base_acts.shape}, ft_acts: {ft_acts.shape}")


## TODO 1 — Compute the finetuning shift

Paper Section 4.2: the **finetuning shift** is the difference between base and
finetuned average activations at layer `L`, projected onto the unit persona
vector:

$$\text{shift} = \frac{v \cdot (\bar{h}_{\text{ft}} - \bar{h}_{\text{base}})}{\|v\|}$$


In [ ]:
# TODO: compute the scalar finetuning shift.
finetune_shift = ...  # YOUR CODE HERE
print(f"finetuning shift along {TRAIT} vector: {finetune_shift:.3f}")


## Step 3 — Measure post-finetune trait expression


In [ ]:
responses = []
for p in eval_prompts:
    responses.append(ft_model.generate(p, max_new_tokens=200, temperature=0.7)[0])
trait_scores = [score_trait(r, TRAIT) for r in responses]
mean_trait = float(np.mean(trait_scores))
print(f"post-FT mean trait score: {mean_trait:.1f}")


## Reproduce Figure 6 (cross-dataset)

Repeat the above for each dataset/variant you've finetuned on and plot
finetune_shift vs. mean_trait. Each dot is one finetune. Paper correlations:
r = 0.76–0.97.


In [ ]:
# Fill this dict from prior runs you've done:
#   {'medical_II': (shift, trait_score), 'code_II': (shift, trait_score), ...}
RESULTS = {
    'medical_II': (finetune_shift, mean_trait),
    # 'code_II': (...),
    # 'gsm8k_II': (...),
}
shifts = np.array([s for s, _ in RESULTS.values()])
traits = np.array([t for _, t in RESULTS.values()])
plt.scatter(shifts, traits)
for name, (s, t) in RESULTS.items():
    plt.annotate(name, (s, t), fontsize=8)
plt.xlabel(f"Finetuning shift along {TRAIT} vector")
plt.ylabel("Post-FT trait score")
plt.title(f"Figure 6 reproduction ({TRAIT})")
plt.grid(alpha=0.3)
plt.show()


## Analysis

1. Did you observe positive correlation? If only one dataset was finetuned, the
   scatter has one point — that's fine for this class; describe the trend
   qualitatively for other domains.
2. The paper shows that even training on *non-trait-targeted* flawed data (e.g.
   bad medical advice) induces an evil persona shift. Reflect on why.
3. This cell's `finetune_shift` uses only 15 eval prompts. Is that enough? How
   would you estimate uncertainty?
